# Clase 3 · 04 · Grafos: clientes que comparten dispositivos

En un grafo el valor está en las **relaciones**. Modelamos un grafo de propiedades como en Neo4j:

```text
(:Customer {id, country, flagged}) -[:USES {tx_count}]-> (:Device {id})
```

Un cliente queda **marcado** (`flagged`) si tiene 3 o más transacciones con `is_fraud = 1`. Pregunta de negocio: *¿qué clientes están conectados, a través de dispositivos compartidos, con clientes marcados?*

Neo4j guarda en cada nodo punteros a sus vecinos (*index-free adjacency*). Acá guardamos el grafo como dos tablas, **nodos** y **aristas**, y cada salto será un **join**.

In [ ]:
dbutils.widgets.text("student_id", "alumno01", "01 - Identificador")
dbutils.widgets.dropdown("scale", "small", ["test", "small", "demo"], "02 - Escala")

In [ ]:
%run ../common/config

In [ ]:
from pyspark.sql import functions as F
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

catalog = spark.sql("SELECT current_catalog()").first()[0]
config = CourseConfig(catalog, dbutils.widgets.get("student_id"), dbutils.widgets.get("scale"))
create_course_namespace(spark, config)

customers = qualified_table(config, "silver_customers")
products = qualified_table(config, "silver_products")
transactions = qualified_table(config, "silver_transactions")
for table in [customers, products, transactions]:
    assert spark.catalog.tableExists(table), f"Falta {table}. Ejecutá 00_preflight (y la clase 2)."
lab_path = f"{config.volume_path}/nosql_lab"
print(f"Esquema: {config.catalog}.{config.schema} · escala {config.scale}")

## 1. Nodos y aristas

In [ ]:
nodes_table = qualified_table(config, "nosql_graph_nodes")
edges_table = qualified_table(config, "nosql_graph_edges")
FLAG_THRESHOLD = 3

fraud_by_customer = (spark.table(transactions).groupBy("customer_id")
    .agg(F.sum("is_fraud").alias("fraud_transactions")))

edges = (spark.table(transactions)
    .select(F.concat(F.lit("c:"), "customer_id").alias("src"), F.concat(F.lit("d:"), "device_id").alias("dst"))
    .groupBy("src", "dst").agg(F.count("*").alias("tx_count"))
    .withColumn("rel", F.lit("USES")))

customer_nodes = (spark.table(customers).join(fraud_by_customer, "customer_id", "left")
    .select(F.concat(F.lit("c:"), "customer_id").alias("id"), F.lit("Customer").alias("label"), "country",
            (F.coalesce("fraud_transactions", F.lit(0)) >= FLAG_THRESHOLD).alias("flagged")))
device_nodes = (edges.select(F.col("dst").alias("id")).distinct()
    .select("id", F.lit("Device").alias("label"), F.lit(None).cast("string").alias("country"), F.lit(False).alias("flagged")))

customer_nodes.unionByName(device_nodes).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(nodes_table)
edges.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(edges_table)

nodes, edges = spark.table(nodes_table), spark.table(edges_table)
display(nodes.groupBy("label").agg(F.count("*").alias("nodos"), F.sum(F.col("flagged").cast("int")).alias("marcados")))
print(f"{edges.count():,} aristas USES")

## 2. Grado de los nodos

El **grado** es la cantidad de vecinos. Un dispositivo usado por muchos clientes es un patrón típico de fraude (granjas de cuentas, dispositivos robados).

In [ ]:
device_degree = edges.groupBy("dst").agg(F.countDistinct("src").alias("customers"))
customer_degree = edges.groupBy("src").agg(F.countDistinct("dst").alias("devices"))
dd = device_degree.groupBy("customers").count().orderBy("customers").toPandas()
cd = customer_degree.groupBy("devices").count().orderBy("devices").toPandas()

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
axes[0].bar(dd["customers"].astype(str), dd["count"], color="#64748b")
axes[0].set_title("¿Cuántos clientes usan cada dispositivo?")
axes[0].set_xlabel("Clientes por dispositivo")
axes[0].set_ylabel("Dispositivos")
axes[1].bar(cd["devices"].astype(str), cd["count"], color="#2563eb")
axes[1].set_title("¿Cuántos dispositivos usa cada cliente?")
axes[1].set_xlabel("Dispositivos por cliente")
axes[1].set_ylabel("Clientes")
fig.tight_layout()
plt.show()

## 3. Un patrón de dos saltos: Cypher vs SQL

En Neo4j la consulta se escribe dibujando el patrón:

```cypher
MATCH (f:Customer {flagged: true})-[:USES]->(d:Device)<-[:USES]-(c:Customer)
WHERE c <> f
RETURN DISTINCT c
```

En SQL, cada flecha del patrón se convierte en un join contra la tabla de aristas:

In [ ]:
nodes.createOrReplaceTempView("graph_nodes")
edges.createOrReplaceTempView("graph_edges")

exposed = spark.sql("""
SELECT DISTINCT c.id, c.flagged
FROM graph_nodes f
JOIN graph_edges e1 ON e1.src = f.id                         -- (f)-[:USES]->(d)
JOIN graph_edges e2 ON e2.dst = e1.dst AND e2.src <> f.id     -- (d)<-[:USES]-(c)
JOIN graph_nodes c  ON c.id = e2.src
WHERE f.flagged
""")

base_rate = nodes.where("label = 'Customer'").agg(F.avg(F.col("flagged").cast("int"))).first()[0]
exposed_rate = exposed.agg(F.avg(F.col("flagged").cast("int"))).first()[0] or 0.0
print(f"Clientes conectados a un marcado por un dispositivo: {exposed.count():,}")

fig, ax = plt.subplots(figsize=(6.5, 3.2))
ax.bar(["todos los clientes", "conectados a un marcado"], [100 * base_rate, 100 * exposed_rate], color=["#94a3b8", "#dc2626"])
ax.set_ylabel("% de clientes marcados")
ax.set_title("¿Compartir dispositivo con un marcado aumenta el riesgo?")
for i, v in enumerate([100 * base_rate, 100 * exposed_rate]):
    ax.text(i, v, f"{v:.1f}%", ha="center", va="bottom")
plt.show()

## 4. Muchos saltos: componentes conexos

Un **componente conexo** agrupa todos los nodos alcanzables entre sí, con cualquier cantidad de saltos: en fraude, una posible "red" de cuentas. Lo calculamos con **propagación de etiquetas**: cada nodo empieza con su propio id como etiqueta y en cada iteración adopta la etiqueta mínima de sus vecinos. Cada iteración es un join, o sea, un salto más.

Este es un procesamiento **OLAP de grafos** (batch sobre todo el grafo), distinto de una consulta **OLTP** de Neo4j que parte de un nodo y recorre pocos vecinos.

In [ ]:
undirected = (edges.select(F.col("src").alias("a"), F.col("dst").alias("b"))
              .unionByName(edges.select(F.col("dst").alias("a"), F.col("src").alias("b"))))
work = [qualified_table(config, "nosql_graph_cc_a"), qualified_table(config, "nosql_graph_cc_b")]
nodes.select("id", F.col("id").alias("component")).write.mode("overwrite").saveAsTable(work[0])

history, current = [], 0
for iteration in range(1, 51):
    labels = spark.table(work[current])
    neighbor_min = (undirected.join(labels.withColumnRenamed("id", "b"), "b")
                    .groupBy("a").agg(F.min("component").alias("neighbor_component"))
                    .withColumnRenamed("a", "id"))
    updated = (labels.join(neighbor_min, "id", "left")
               .select("id", F.least("component", F.coalesce("neighbor_component", "component")).alias("component")))
    updated.write.mode("overwrite").saveAsTable(work[1 - current])
    current = 1 - current
    changed = (spark.table(work[current]).alias("new").join(labels.alias("old"), "id")
               .where("new.component <> old.component").count())
    history.append((iteration, changed))
    print(f"iteración {iteration}: {changed:,} nodos cambiaron de componente")
    if changed == 0:
        break

components_table = qualified_table(config, "nosql_graph_components")
spark.table(work[current]).write.mode("overwrite").saveAsTable(components_table)
components = spark.table(components_table)

In [ ]:
sizes = components.groupBy("component").agg(F.count("*").alias("size"))
size_dist = sizes.groupBy("size").count().orderBy("size").toPandas()
hist = pd.DataFrame(history, columns=["iteración", "cambios"])

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
axes[0].plot(hist["iteración"], hist["cambios"], marker="o", color="#2563eb")
axes[0].set_title("Convergencia de la propagación de etiquetas")
axes[0].set_xlabel("Iteración (= un salto más)")
axes[0].set_ylabel("Nodos que cambiaron")
axes[1].bar(size_dist["size"].astype(str), size_dist["count"], color="#7c3aed")
axes[1].set_yscale("log")
axes[1].set_title("Tamaño de los componentes")
axes[1].set_xlabel("Nodos en el componente")
axes[1].set_ylabel("Componentes (escala log)")
fig.tight_layout()
plt.show()
display(sizes.orderBy(F.desc("size")).limit(5))

## 5. Dibujar el componente más grande

Rojo: clientes marcados. Azul: otros clientes. Cuadrados grises: dispositivos.

In [ ]:
def spring_layout(node_ids, pairs, iterations=300, seed=7):
    """Layout de fuerzas (Fruchterman-Reingold) en numpy, suficiente para grafos chicos."""
    rng = np.random.default_rng(seed)
    index = {v: i for i, v in enumerate(node_ids)}
    pos = rng.uniform(-1, 1, (len(node_ids), 2))
    e = np.array([(index[a], index[b]) for a, b in pairs])
    k, step = np.sqrt(4.0 / len(node_ids)), 0.1
    for _ in range(iterations):
        delta = pos[:, None, :] - pos[None, :, :]
        dist = np.linalg.norm(delta, axis=-1) + 1e-9
        disp = (delta / dist[..., None] * (k * k / dist)[..., None]).sum(axis=1)
        d = pos[e[:, 0]] - pos[e[:, 1]]
        dl = np.linalg.norm(d, axis=1) + 1e-9
        force = d / dl[:, None] * (dl * dl / k)[:, None]
        np.add.at(disp, e[:, 0], -force)
        np.add.at(disp, e[:, 1], force)
        length = np.linalg.norm(disp, axis=1) + 1e-9
        pos += disp / length[:, None] * np.minimum(length, step)[:, None]
        step *= 0.98
    return pos

largest = sizes.orderBy(F.desc("size"), "component").first()["component"]
member_ids = components.where(F.col("component") == largest).select("id")
sub_nodes = nodes.join(member_ids, "id").limit(150).toPandas()
keep = set(sub_nodes["id"])
sub_edges = [(r.src, r.dst) for r in edges.where(F.col("src").isin(list(keep)) & F.col("dst").isin(list(keep))).collect()]
pos = spring_layout(list(sub_nodes["id"]), sub_edges)
xy = dict(zip(sub_nodes["id"], pos))

fig, ax = plt.subplots(figsize=(9, 7))
for a, b in sub_edges:
    ax.plot(*zip(xy[a], xy[b]), color="#cbd5e1", lw=1, zorder=1)
for row in sub_nodes.itertuples():
    x, y = xy[row.id]
    if row.label == "Device":
        ax.scatter(x, y, marker="s", s=60, color="#64748b", zorder=2)
    else:
        ax.scatter(x, y, s=140, color="#dc2626" if row.flagged else "#2563eb", zorder=3)
    ax.annotate(row.id.split(":", 1)[1].replace("device_", "d"), (x, y), fontsize=7, ha="center", va="bottom")
ax.set_title(f"Componente más grande: {len(sub_nodes)} nodos, {len(sub_edges)} aristas")
ax.axis("off")
plt.show()

## 6. ¿Por qué cuesta distribuir un grafo?

Si repartimos los nodos entre 4 máquinas con `hash(id) mod 4` (como en clave-valor), cada arista cuyos extremos quedan en máquinas distintas obliga a un salto por la red. Comparamos con un particionamiento que mantiene juntos los nodos del mismo componente.

In [ ]:
N_PARTITIONS = 4
assignments = (components
    .withColumn("by_hash", F.pmod(F.hash("id"), F.lit(N_PARTITIONS)))
    .withColumn("by_component", F.pmod(F.hash("component"), F.lit(N_PARTITIONS))))
a_src = assignments.select(F.col("id").alias("src"), F.col("by_hash").alias("hs"), F.col("by_component").alias("cs"))
a_dst = assignments.select(F.col("id").alias("dst"), F.col("by_hash").alias("hd"), F.col("by_component").alias("cd"))
cut = (edges.join(a_src, "src").join(a_dst, "dst")
    .agg(F.avg((F.col("hs") != F.col("hd")).cast("int")).alias("hash"),
         F.avg((F.col("cs") != F.col("cd")).cast("int")).alias("component"))).first()
load = assignments.groupBy("by_hash").count().toPandas().sort_values("by_hash")
load_c = assignments.groupBy("by_component").count().toPandas().sort_values("by_component")

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
axes[0].bar(["hash(id) mod 4", "por componente"], [100 * cut["hash"], 100 * cut["component"]], color=["#dc2626", "#16a34a"])
axes[0].set_ylabel("% de aristas entre máquinas")
axes[0].set_title("Aristas cortadas por el particionamiento")
for i, v in enumerate([100 * cut["hash"], 100 * cut["component"]]):
    axes[0].text(i, v, f"{v:.1f}%", ha="center", va="bottom")
x = np.arange(N_PARTITIONS)
axes[1].bar(x - 0.2, load["count"], 0.4, label="hash(id)", color="#fca5a5")
axes[1].bar(x + 0.2, load_c["count"], 0.4, label="por componente", color="#86efac")
axes[1].set_xticks(x, [f"máquina {i}" for i in x])
axes[1].set_ylabel("Nodos")
axes[1].set_title("Carga por máquina")
axes[1].legend(loc="lower center")
fig.tight_layout()
plt.show()

## 7. Más saltos, más joins

El patrón de **cuatro saltos** en Cypher sigue siendo un dibujo:

```cypher
MATCH (f:Customer {flagged: true})-[:USES]->(:Device)<-[:USES]-(:Customer)-[:USES]->(:Device)<-[:USES]-(c:Customer)
RETURN DISTINCT c
```

En SQL cada flecha agrega un join contra `graph_edges`:

In [ ]:
four_hops = spark.sql("""
SELECT DISTINCT c.id
FROM graph_nodes f
JOIN graph_edges e1 ON e1.src = f.id
JOIN graph_edges e2 ON e2.dst = e1.dst AND e2.src <> f.id
JOIN graph_edges e3 ON e3.src = e2.src AND e3.dst <> e1.dst
JOIN graph_edges e4 ON e4.dst = e3.dst AND e4.src <> e2.src
JOIN graph_nodes c  ON c.id = e4.src
WHERE f.flagged
""")
display(pd.DataFrame([
    {"saltos": 2, "joins en SQL": 3, "clientes alcanzados": exposed.count()},
    {"saltos": 4, "joins en SQL": 5, "clientes alcanzados": four_hops.count()},
]))